# Creating an Index

This jupyter notebook creates a FAISS based index consisting of docstrings of [SciKit Learn]() esitmators. This index is then used to simulate how vector database and LLM models can be used and integrated into DataRobot. 

As a first step, let's gather all of the docstrings from Scikit-Learn. 


In [1]:
import inspect
import os
import json
import numpy as np
import time
from sklearn.utils.discovery import all_estimators

In [2]:
estimators = all_estimators()

docstrings = []
class_names = []

for name, estimator in estimators:
    # Check if it's actually an estimator (has fit method)
    if hasattr(estimator, 'fit') and inspect.isclass(estimator):
        doc = estimator.__doc__
        if doc is not None and len(doc.strip()) > 0:
            docstrings.append(doc)
            class_names.append(name)
            
print(f"Extracted {len(docstrings)} docstrings")
print(f"Here is a sample of {class_names[0]} \n")
print(docstrings[0][0:200])

Extracted 207 docstrings
Here is a sample of ARDRegression 

Bayesian ARD regression.

    Fit the weights of a regression model, using an ARD prior. The weights of
    the regression model are assumed to be in Gaussian distributions.
    Also estimate the para


In [3]:
# Create the embeddings
from transformers import AutoTokenizer, AutoModel
import torch
import os 
import time


MODEL_DIR = "embedding_model"
model_name = "prajjwal1/bert-tiny"

os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Loading model from/to: {MODEL_DIR}")
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=MODEL_DIR)
model = AutoModel.from_pretrained(model_name, cache_dir=MODEL_DIR).cpu()

device = torch.device("cpu")

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # First element contains token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Process in batches to avoid memory issues
batch_size = 64  # Larger batch size for speed
embeddings_list = []

start_time = time.time()

for i in range(0, len(docstrings), batch_size):
    batch = docstrings[i:i+batch_size]
    
    # Tokenize
    encoded_input = tokenizer(batch, padding=True, truncation=True, 
                             max_length=256, return_tensors='pt')  # Reduced max_length for speed
    
    # Compute token embeddings
    with torch.no_grad():
        model_output = model(**encoded_input)
    
    # Apply mean pooling
    batch_embeddings = mean_pooling(model_output, encoded_input['attention_mask']).numpy()
    embeddings_list.append(batch_embeddings)
    
    # Show progress sparingly for speed
    if (i // batch_size) % 20 == 0:
        print(f"Processed {i+len(batch)}/{len(docstrings)} documents...")

# Concatenate all batches
embeddings = np.vstack(embeddings_list)

elapsed_time = time.time() - start_time

print(f"Created {embeddings.shape[1]}-dimensional embeddings for {len(docstrings)} documents")
print(f"Processing took {elapsed_time:.2f} seconds ({len(docstrings)/elapsed_time:.2f} docs/sec)")
    

Loading model from/to: embedding_model
Processed 64/207 documents...
Created 128-dimensional embeddings for 207 documents
Processing took 0.34 seconds (600.87 docs/sec)


In [4]:
import chromadb
from chromadb.utils import embedding_functions
client = chromadb.Client()
persistent_client = chromadb.PersistentClient(path="./chroma_db")
persistent_collection = persistent_client.create_collection("sklearn")

documents = docstrings # Your original texts
ids = class_names  

persistent_collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),  # Convert numpy to list
    ids=ids
)

# Save collection (persistent storage)
# ChromaDB handles persistence through its PersistentClient



## Test a Query

In [6]:
query = "clustering algorithm for large datasets"

# Initialize a persistent ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")
collection_name = "sklearn"

collection = client.get_collection(collection_name)
print(f"Loaded existing collection: {collection_name}")


encoded_input = tokenizer([query], padding=True, truncation=True, 
                             max_length=256, return_tensors='pt')
    
with torch.no_grad():
    model_output = model(**encoded_input)

token_embeddings = model_output[0]
input_mask_expanded = encoded_input['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
query_embedding = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
query_embedding = query_embedding.numpy()

# Query the collection
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)

# Display results
print(f"\nTop 3 results for query: '{query}'")
print(results)
for i in range(len(results['ids'][0])):
    doc_id = results['ids'][0][i]
    distance = results['distances'][0][i]
    document = results['documents'][0][i]

    class_name = doc_id
    
    print(f"{i+1}. {class_name} (Distance: {distance:.4f})")
    print(f"   {document[:150]}...\n")
    
    
    print(f"{i+1}. {class_name} (Distance: {distance:.4f})")
    print(f"   {document[:150]}...\n")

Loaded existing collection: sklearn

Top 3 results for query: 'clustering algorithm for large datasets'
{'ids': [['OPTICS', 'EllipticEnvelope', 'IsolationForest']], 'embeddings': None, 'documents': [['Estimate clustering structure from vector array.\n\n    OPTICS (Ordering Points To Identify the Clustering Structure), closely\n    related to DBSCAN, finds core sample of high density and expands clusters\n    from them [1]_. Unlike DBSCAN, keeps cluster hierarchy for a variable\n    neighborhood radius. Better suited for usage on large datasets than the\n    current sklearn implementation of DBSCAN.\n\n    Clusters are then extracted using a DBSCAN-like method\n    (cluster_method = \'dbscan\') or an automatic\n    technique proposed in [1]_ (cluster_method = \'xi\').\n\n    This implementation deviates from the original OPTICS by first performing\n    k-nearest-neighborhood searches on all points to identify core sizes, then\n    computing only the distances to unprocessed points when 